# Week 56, Day 4: Decision theory: losses, thresholds and ROC curves

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w056/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The threshold that minimises cost

Write total_cost(p, t, thr, c_fp, c_fn). For calibrated probabilities on 20,000 test trials, find the threshold on a grid that minimises cost when a false negative costs k times a false positive, for k = 1, 4 and 9, and compare with the Bayes threshold 1 / (1 + k).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(564)
n = 20000
score = r.normal(size=n)
p_true = 1 / (1 + np.exp(-(1.5 * score - 0.5)))       # calibrated probabilities
t = (r.random(n) < p_true).astype(int)

def total_cost(p, t, thr, c_fp, c_fn):
    say1 = p > thr
    return ...

grid = np.linspace(0.01, 0.99, 99)
best = {}
for k in [1, 4, 9]:
    costs = [total_cost(p_true, t, g, 1.0, k) for g in grid]
    best[k] = grid[int(np.argmin(costs))]
    print(f"false negatives cost {k}x: best threshold {best[k]:.2f}, Bayes threshold {1 / (1 + k):.2f}")

In [ ]:
import numpy as np
def test_cost():
    "total_cost counts false positives times c_fp plus false negatives times c_fn"
    t = np.array([1, 0, 1, 0]); p = np.array([0.9, 0.8, 0.2, 0.1])
    assert total_cost(p, t, 0.5, 1.0, 5.0) == 1 * 1.0 + 1 * 5.0
def test_best():
    "the empirically best threshold is within 0.05 of the Bayes threshold for each cost ratio"
    assert all(abs(best[k] - 1 / (1 + k)) < 0.05 for k in best)

test_cost()
test_best()
print("All checks passed")

### Exercise: ROC from scratch

Write roc_curve(scores, labels): sort trials by decreasing score and accumulate true and false positive rates, starting from (0, 0). Compute the area by the trapezoid rule, the fraction of positive-negative pairs ranked correctly (on a subsample), and the theoretical value for d' = 1.2.

In [ ]:
from scipy.stats import norm
dp = 1.2
lab = r.integers(0, 2, 4000)
sc = r.normal(size=4000) + dp * lab

def roc_curve(scores, labels):
    order = np.argsort(-scores)
    l = labels[order]
    tpr = np.r_[0, np.cumsum(l) / l.sum()]
    fpr = ...
    return fpr, tpr

fpr, tpr = roc_curve(sc, lab)
auc_trap = np.trapz(tpr, fpr) if hasattr(np, "trapz") else np.trapezoid(tpr, fpr)
pos, neg = sc[lab == 1][:800], sc[lab == 0][:800]
auc_pairs = ...
auc_theory = norm.cdf(dp / np.sqrt(2))
print(f"AUC: trapezoid {auc_trap:.3f}, pairwise {auc_pairs:.3f}, theory {auc_theory:.3f}")
plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "k:"); plt.xlabel("false positive rate"); plt.ylabel("true positive rate"); plt.show()

In [ ]:
import numpy as np
def test_roc():
    "roc_curve returns rates from (0, 0) to (1, 1), non-decreasing"
    f, tp = roc_curve(np.array([0.1, 0.4, 0.35, 0.8]), np.array([0, 0, 1, 1]))
    assert f[0] == 0 and tp[0] == 0 and f[-1] == 1 and tp[-1] == 1 and np.all(np.diff(f) >= 0)
def test_auc():
    "the trapezoid AUC matches the pairwise probability and the d' formula within 0.01"
    assert abs(auc_trap - auc_pairs) < 0.01 and abs(auc_trap - auc_theory) < 0.01

test_roc()
test_auc()
print("All checks passed")

### Exercise: Scores that see through imbalance

Write metrics(pred, t) returning accuracy, balanced accuracy, precision and recall. Score the always-no detector on data with 5% events, then score a real detector (threshold 1.5 on a score with d' = 2) at 5% and at 30% prevalence.

In [ ]:
def metrics(pred, t):
    tp, tn = np.sum(pred & (t == 1)), np.sum(~pred & (t == 0))
    fp, fn = np.sum(pred & (t == 0)), np.sum(~pred & (t == 1))
    return {"accuracy": (tp + tn) / len(t), "balanced": ...,
            "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn)}

def events(n, prev, g):
    t_ = (g.random(n) < prev).astype(int)
    return g.normal(size=n) + 2.0 * t_, t_

s5, t5 = events(20000, 0.05, r); s30, t30 = events(20000, 0.30, r)
scores_triv = metrics(np.zeros(20000, bool), t5)
m5, m30 = metrics(s5 > 1.5, t5), metrics(s30 > 1.5, t30)
prec_rare, rec_rare, prec_common, rec_common = m5["precision"], m5["recall"], m30["precision"], m30["recall"]
print("always no, 5% events:", {k: round(float(v), 3) for k, v in scores_triv.items()})
print("detector, 5% events: ", {k: round(float(v), 3) for k, v in m5.items()})
print("detector, 30% events:", {k: round(float(v), 3) for k, v in m30.items()})

In [ ]:
import numpy as np
def test_trivial():
    "the always-no detector has accuracy near 0.95 and balanced accuracy 0.5"
    assert abs(scores_triv["accuracy"] - 0.95) < 0.01 and abs(scores_triv["balanced"] - 0.5) < 1e-9
def test_precision():
    "at the same threshold, precision falls when events become rarer while recall stays the same"
    assert prec_rare < prec_common - 0.2 and abs(rec_rare - rec_common) < 0.03

test_trivial()
test_precision()
print("All checks passed")